In [1]:
import time
import json
from typing import TypedDict, Optional
from openai import OpenAI
from langgraph.graph import StateGraph

# 🔑 Add your API key
client = OpenAI(api_key="YOUR_OPENAI_API_KEY_HERE")

ModuleNotFoundError: No module named 'openai'

In [ ]:
knowledge_base = {
    "pricing": {
        "basic": "Basic Plan: $29/month, 10 videos/month, 720p resolution",
        "pro": "Pro Plan: $79/month, Unlimited videos, 4K resolution, AI captions"
    },
    "policies": {
        "refund": "No refunds after 7 days",
        "support": "24/7 support available only on Pro plan"
    }
}

In [ ]:
def safe_llm_call(prompt):
    for i in range(5):  # retry 5 times
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7
            )
            return response.choices[0].message.content
        except Exception as e:
            print("⚠️ Rate limit hit, retrying...", i+1)
            time.sleep(2 ** i)  # exponential backoff

    return "⚠️ AI is busy right now. But I can still help with pricing and signup."

In [ ]:
def detect_intent(user_input):
    text = user_input.lower()

    if any(word in text.lower() for word in ["hi", "hello", "hey","hy"]):
        print(f"Detected intent: greeting for input: {user_input}") # Debug print
        return "greeting"

    # Prioritize high_intent over pricing if keywords suggest strong interest
    if any(word in text.lower() for word in ["buy", "subscribe", "start", "try", "sign up", "like", "interested"]):
        print(f"Detected intent: high_intent for input: {user_input}") # Debug print
        return "high_intent"

    if any(phrase in text.lower() for phrase in ["price", "plan", "cost", "pricing plans", "tell me about your plans", "what are your plans", "how much", "cost of"]):
        print(f"Detected intent: pricing for input: {user_input}") # Debug print
        return "pricing"

    if any(word in text.lower() for word in ["refund", "policy", "support"]):
        print(f"Detected intent: policy for input: {user_input}") # Debug print
        return "policy"

    print(f"Detected intent: general for input: {user_input}") # Debug print
    return "general"

In [ ]:
def mock_lead_capture(name, email, platform):
    print(f"\n✅ Lead captured successfully: {name}, {email}, {platform}\n")

In [ ]:
class AgentState(TypedDict):
    messages: list
    name: Optional[str]
    email: Optional[str]
    platform: Optional[str]
    intent: Optional[str]

In [ ]:
def rag_response(state):
    user_input = state["messages"][-1]

    if state["intent"] == "pricing":
        return (
            knowledge_base["pricing"]["basic"] + "\n" +
            knowledge_base["pricing"]["pro"]
        )

    if state["intent"] == "policy":
        return (
            knowledge_base["policies"]["refund"] + "\n" +
            knowledge_base["policies"]["support"]
        )

    if state["intent"] == "greeting":
        return "Hi! Ask me anything about AutoStream 😊"

    if state["intent"] == "general":
        return safe_llm_call(user_input)

    return ""

In [ ]:
def lead_flow(state, user_input):

    if not state["lead_mode"]:
        return None

    # Step 1: Ask Name
    if state["name"] is None:
        state["name"] = user_input
        return "Great! Please share your email."

    # Step 2: Ask Email
    if state["email"] is None:
        state["email"] = user_input
        return "Which platform do you create content on? (YouTube/Instagram/Facebook)"

    # Step 3: Ask Platform
    if state["platform"] is None:
        state["platform"] = user_input

        # Call tool ONLY here
        mock_lead_capture(state["name"], state["email"], state["platform"])

        state["lead_mode"] = False  # reset

        return "🎉 You're all set! Our team will reach out soon."

    return None

In [ ]:
state = {
    "messages": [],
    "name": None,
    "email": None,
    "platform": None,
    "intent": None,
    "lead_mode": False
}

print("🤖 AutoStream Agent Ready!")

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit","stop"]:
        break

    state["messages"].append(user_input)

    # Detect intent ONLY if not in lead mode
    if not state["lead_mode"]:
        state["intent"] = detect_intent(user_input)

    # Handle entering lead mode for the first time or if already in lead mode
    if state["intent"] == "high_intent" and not state["lead_mode"]:
        state["lead_mode"] = True
        print("Bot: Great choice! Let's get you started. What's your name?")
        continue  # Skip the rest of this iteration to avoid processing the high-intent phrase as a name

    # If already in lead mode, process the user's input for lead details
    if state["lead_mode"]:
        lead_response = lead_flow(state, user_input)

        if lead_response:
            print("Bot:", lead_response)
            continue   # ✅ VERY IMPORTANT (stops API call)

    # RAG / LLM response
    response = rag_response(state)
    print("Bot:", response)

🤖 AutoStream Agent Ready!
You: Hi
Detected intent: greeting for input: Hi
Bot: Hi! Ask me anything about AutoStream 😊
You: Plans
Detected intent: pricing for input: Plans
Bot: Basic Plan: $29/month, 10 videos/month, 720p resolution
Pro Plan: $79/month, Unlimited videos, 4K resolution, AI captions
You: Which plan is better for me
Detected intent: greeting for input: Which plan is better for me
Bot: Hi! Ask me anything about AutoStream 😊
You: exit
